In [1]:
!pip install gliner2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 102.9 MB/s eta 0:00:0000:0100:01


In [3]:
from gliner2 import GLiNER2

# Load the model
extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")

# Extract entities
# text = "Apple CEO Tim Cook announced iPhone 15 in Cupertino yesterday."
text = "In a joint announcement from their regional innovation hub in Singapore, QuantumVista Analytics confirmed that its CEO, Dr. Elena Markovic, will oversee the pilot deployment of the AuroraX Edge platform, a next-generation industrial IoT product developed in collaboration with NexaForge Systems. The rollout, coordinated with logistics partner BlueHarbor Freight, will initially target manufacturing sites in Stuttgart, Germany and later expand to Austin, Texas, where CTO Marcus Liu previously led R&D efforts for the now-discontinued Helios One device. Although several media outlets speculated that Orion Dynamics might acquire the AuroraX Edge line, Markovic clarified during a press briefing in Zurich that QuantumVista Analytics intends to retain full ownership while exploring distribution agreements through its subsidiary, Polaris Grid Technologies, headquartered in Toronto, Canada."

result = extractor.extract_entities(text, ["company", "person", "product", "location"])

print(result)
# Output: {'entities': {'company': ['Apple'], 'person': ['Tim Cook'], 'product': ['iPhone 15'], 'location': ['Cupertino']}}


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
{'entities': {'company': ['QuantumVista Analytics', 'Polaris Grid Technologies', 'BlueHarbor Freight', 'Orion Dynamics', 'NexaForge Systems'], 'person': ['Marcus Liu', 'Elena Markovic'], 'product': ['Helios One', 'AuroraX Edge'], 'location': ['Toronto', 'Singapore', 'Austin', 'Stuttgart', 'Zurich', 'Germany', 'Canada', 'Texas']}}


In [4]:
# Single-label classification
result = extractor.classify_text(
    "This laptop has amazing performance but terrible battery life!",
    {"sentiment": ["positive", "negative", "neutral"]}
)
print(result)
# Output: {'sentiment': 'negative'}

# Multi-label classification
result = extractor.classify_text(
    "Great camera quality, decent performance, but poor battery life.",
    {
        "aspects": {
            "labels": ["camera", "performance", "battery", "display", "price"],
            "multi_label": True,
            "cls_threshold": 0.4
        }
    }
)
print(result)
# Output: {'aspects': ['camera', 'performance', 'battery']}


{'sentiment': 'negative'}
{'aspects': ['camera', 'performance', 'battery']}


In [6]:
# Combine all extraction types
schema = (extractor.create_schema()
    .entities({
        "person": "Names of people or individuals",
        "company": "Organization or business names",
        "product": "Products or services mentioned"
    })
    .classification("sentiment", ["positive", "negative", "neutral"])
    .structure("product_info")
        .field("name", dtype="str")
        .field("price", dtype="str")
        .field("features", dtype="list")
)

text = "In a joint announcement from their regional innovation hub in Singapore, QuantumVista Analytics confirmed that its CEO, Dr. Elena Markovic, will oversee the pilot deployment of the AuroraX Edge platform, a next-generation industrial IoT product developed in collaboration with NexaForge Systems. The rollout, coordinated with logistics partner BlueHarbor Freight, will initially target manufacturing sites in Stuttgart, Germany and later expand to Austin, Texas, where CTO Marcus Liu previously led R&D efforts for the now-discontinued Helios One device. Although several media outlets speculated that Orion Dynamics might acquire the AuroraX Edge line, Markovic clarified during a press briefing in Zurich that QuantumVista Analytics intends to retain full ownership while exploring distribution agreements through its subsidiary, Polaris Grid Technologies, headquartered in Toronto, Canada. The AuroraX Edge platform, priced at $999, boasts features such as real-time analytics, edge computing capabilities, and seamless integration with existing industrial systems."
results = extractor.extract(text, schema)

print(results)
# Output: {
#     'entities': {'person': ['Tim Cook'], 'company': ['Apple'], 'product': ['iPhone 15 Pro']},
#     'sentiment': 'positive',
#     'product_info': [{'name': 'iPhone 15 Pro', 'price': '$999', 'features': [...]}]
# }


{'product_info': [{'name': 'AuroraX Edge platform', 'price': '$999', 'features': ['edge computing capabilities', 'real-time analytics', 'seamless integration with existing industrial systems']}], 'entities': {'person': ['Marcus Liu', 'Dr. Elena Markovic'], 'company': ['QuantumVista Analytics', 'Polaris Grid Technologies', 'BlueHarbor Freight', 'Orion Dynamics', 'NexaForge Systems'], 'product': ['AuroraX Edge platform', 'Helios One device']}, 'sentiment': 'positive'}


### Here is the way to do the evaluations for the model tasks

In [7]:
"""
Test structure extraction with include_confidence and include_spans parameters.

This test demonstrates the new API features for structured data extraction.
"""

import json
from gliner2 import GLiNER2


def test_structure_extraction():
    """Test structure extraction with various output formats."""
    
    print("=" * 80)
    print("STRUCTURE EXTRACTION TESTS")
    print("=" * 80)
    
    print("\nLoading model: fastino/gliner2-base-v1...")
    model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
    print("Model loaded successfully!\n")
    
    text = "Apple announced a new iPhone 15 Pro Max at $1099 during their September event in Cupertino."
    
    print(f"\nTest Text: {text}")
    print("\n" + "-" * 80)
    
    # Define schema
    schema = model.create_schema()
    schema.structure("product_announcement")\
        .field("company")\
        .field("product")\
        .field("price")\
        .field("date")\
        .field("location")
    
    print("\nSchema: product_announcement")
    print("  Fields: company, product, price, date, location")
    
    # Test 1: Basic extraction (default)
    print("\n1. BASIC EXTRACTION (text only)")
    print("-" * 40)
    result = model.extract(text, schema)
    print(json.dumps(result, indent=2))
    
    # Test 2: With confidence scores
    print("\n2. WITH CONFIDENCE SCORES")
    print("-" * 40)
    result = model.extract(text, schema, include_confidence=True)
    print(json.dumps(result, indent=2))
    
    # Test 3: With span positions
    print("\n3. WITH SPAN POSITIONS")
    print("-" * 40)
    result = model.extract(text, schema, include_spans=True)
    print(json.dumps(result, indent=2))
    
    # Test 4: With both confidence and spans
    print("\n4. WITH CONFIDENCE AND SPAN POSITIONS")
    print("-" * 40)
    result = model.extract(text, schema, include_confidence=True, include_spans=True)
    print(json.dumps(result, indent=2))
    
    # Test 5: Verify character positions
    print("\n5. VERIFY CHARACTER POSITIONS")
    print("-" * 40)
    for struct in result["product_announcement"]:
        for field_name, field_values in struct.items():
            print(f"\n{field_name}:")
            for value in field_values:
                extracted_text = text[value["start"]:value["end"]]
                print(f"  '{value['text']}' at [{value['start']}:{value['end']}]")
                print(f"  Verification: '{extracted_text}' - Match: {extracted_text == value['text']}")
    
    print("\n" + "=" * 80)
    print("Structure extraction tests completed!")
    print("=" * 80)


def test_structure_with_single_values():
    """Test structure extraction with single-value fields (dtype='str')."""
    
    print("\n" + "=" * 80)
    print("STRUCTURE EXTRACTION WITH SINGLE VALUES")
    print("=" * 80)
    
    print("\nLoading model: fastino/gliner2-base-v1...")
    model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
    print("Model loaded successfully!\n")
    
    text = "Apple announced iPhone 15 at $999 on September 12."
    
    print(f"\nTest Text: {text}")
    print("\n" + "-" * 80)
    
    # Define schema with dtype='str' for single values
    schema = model.create_schema()
    schema.structure("product_info")\
        .field("company", dtype="str")\
        .field("product", dtype="str")\
        .field("price", dtype="str")\
        .field("date", dtype="str")
    
    print("\nSchema: product_info (all fields dtype='str')")
    print("  Fields: company, product, price, date")
    
    # Test with all flags
    print("\n1. WITH CONFIDENCE AND SPAN POSITIONS")
    print("-" * 40)
    result = model.extract(text, schema, include_confidence=True, include_spans=True)
    print(json.dumps(result, indent=2))
    
    # Test 2: Only spans (no confidence)
    print("\n2. WITH SPAN POSITIONS ONLY")
    print("-" * 40)
    result = model.extract(text, schema, include_spans=True)
    print(json.dumps(result, indent=2))
    
    # Test 3: Basic (no metadata)
    print("\n3. BASIC (text only)")
    print("-" * 40)
    result = model.extract(text, schema)
    print(json.dumps(result, indent=2))
    
    print("\n" + "=" * 80)
    print("Single-value structure tests completed!")
    print("=" * 80)


def test_batch_structure_extraction():
    """Test batch structure extraction."""
    
    print("\n" + "=" * 80)
    print("BATCH STRUCTURE EXTRACTION TESTS")
    print("=" * 80)
    
    print("\nLoading model: fastino/gliner2-base-v1...")
    model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
    print("Model loaded successfully!\n")
    
    texts = [
        "Apple announced iPhone 15 at $999 on September 12.",
        "Google released Pixel 8 for $699 in October.",
        "Microsoft launched Surface Pro 9 at $1299."
    ]
    
    print(f"\nNumber of texts: {len(texts)}")
    
    schema = model.create_schema()
    schema.structure("product_launch")\
        .field("company")\
        .field("product")\
        .field("price")\
        .field("date")
    
    print("\n1. BATCH WITH CONFIDENCE AND SPANS")
    print("-" * 40)
    results = model.batch_extract(
        texts, schema, batch_size=2,
        include_confidence=True, include_spans=True
    )
    
    for i, (text, result) in enumerate(zip(texts, results)):
        print(f"\nText {i+1}: {text}")
        print(f"Result: {json.dumps(result, indent=2)}")
    
    print("\n" + "=" * 80)
    print("Batch structure extraction tests completed!")
    print("=" * 80)


if __name__ == "__main__":
    test_structure_extraction()
    test_structure_with_single_values()
    test_batch_structure_extraction()

STRUCTURE EXTRACTION TESTS

Loading model: fastino/gliner2-base-v1...


You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Model loaded successfully!


Test Text: Apple announced a new iPhone 15 Pro Max at $1099 during their September event in Cupertino.

--------------------------------------------------------------------------------

Schema: product_announcement
  Fields: company, product, price, date, location

1. BASIC EXTRACTION (text only)
----------------------------------------
{
  "product_announcement": [
    {
      "company": [
        "Apple"
      ],
      "product": [
        "iPhone 15 Pro Max"
      ],
      "price": [
        "$1099"
      ],
      "date": [
        "September"
      ],
      "location": [
        "Cupertino"
      ]
    }
  ]
}

2. WITH CONFIDENCE SCORES
----------------------------------------
{
  "product_announcement": [
    {
      "company": [
        {
          "text": "Apple",
          "confidence": 0.9999997615814209
        }
     

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Model loaded successfully!


Test Text: Apple announced iPhone 15 at $999 on September 12.

--------------------------------------------------------------------------------

Schema: product_info (all fields dtype='str')
  Fields: company, product, price, date

1. WITH CONFIDENCE AND SPAN POSITIONS
----------------------------------------
{
  "product_info": [
    {
      "company": {
        "text": "Apple",
        "confidence": 0.9999997615814209,
        "start": 0,
        "end": 5
      },
      "product": {
        "text": "iPhone 15",
        "confidence": 1.0,
        "start": 16,
        "end": 25
      },
      "price": {
        "text": "$999",
        "confidence": 1.0,
        "start": 29,
        "end": 33
      },
      "date": {
        "text": "September 12",
        "confidence": 1.0,
        "start": 37,
        "end": 49
      }
    }
  

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Model loaded successfully!


Number of texts: 3

1. BATCH WITH CONFIDENCE AND SPANS
----------------------------------------

Text 1: Apple announced iPhone 15 at $999 on September 12.
Result: {
  "product_launch": [
    {
      "company": [
        {
          "text": "Apple",
          "confidence": 0.9999998807907104,
          "start": 0,
          "end": 5
        }
      ],
      "product": [
        {
          "text": "iPhone 15",
          "confidence": 1.0,
          "start": 16,
          "end": 25
        }
      ],
      "price": [
        {
          "text": "$999",
          "confidence": 1.0,
          "start": 29,
          "end": 33
        }
      ],
      "date": [
        {
          "text": "September 12",
          "confidence": 1.0,
          "start": 37,
          "end": 49
        }
      ]
    }
  ]
}

Text 2: Google released Pixe